[Simulated Trading Book]
        ↓
[Market Data Layer — yfinance / synthetic]
        ↓
[Risk Engine]
  ├── FRTB SA Capital Calculator (RWA per risk class)
  ├── CCAR Stress Tester (3 scenarios × 5 macro shocks)
  ├── Balance Sheet Optimizer (SciPy LP/QP)
  └── Liquidity Ratio Calculator (LCR / NSFR)
        ↓
[Analytics Layer]
  ├── Performance Attribution
  ├── Capital Efficiency Metrics (RoRC, RWA density)
  └── Regulatory Limit Monitoring
        ↓
[Dashboard — Plotly + Excel Export via openpyxl]



FRM_Project/
│
├── FRM_Engine.ipynb              ← Main Colab notebook
│
├── /data/
│   ├── trading_book.csv          ← Simulated positions
│   ├── market_data.csv           ← Price history
│   └── macro_scenarios.csv       ← CCAR scenarios
│
├── /outputs/
│   ├── frm_report.xlsx           ← Final Excel deliverable
│   ├── capital_dashboard.html    ← Interactive Plotly dashboard
│   └── stress_results.csv        ← CCAR output
│
└── /charts/
    ├── rwa_breakdown.png
    ├── stress_pnl.png
    ├── bs_optimization.png
    └── liquidity_ratios.png


In [13]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP
# Markets FRM Analytics Engine
# Author:Shashank Ravindrakumar | MSc Economics & Finance, KCL
# Date: June 2026
# ============================================================

# Install required packages
!pip install yfinance cvxpy openpyxl plotly pandas-datareader kaleido -q

import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import itertools
import os

# Financial data
import yfinance as yf

# Optimization
from scipy.optimize import linprog, minimize
from scipy.stats import norm
try:
    import cvxpy as cp
    CVXPY_AVAILABLE = True
except ImportError:
    CVXPY_AVAILABLE = False

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Reporting
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.utils.dataframe import dataframe_to_rows

# Setup output directories
os.makedirs('/content/outputs', exist_ok=True)
os.makedirs('/content/charts', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)

# Random seed for reproducibility
np.random.seed(42)

print("=" * 60)
print("  MARKETS FRM ANALYTICS ENGINE - INITIALIZED")
print("  Capital | Balance Sheet | Liquidity | Stress Testing")
print("=" * 60)
print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("  Status: All libraries loaded successfully ✓")
print("=" * 60)

  MARKETS FRM ANALYTICS ENGINE - INITIALIZED
  Capital | Balance Sheet | Liquidity | Stress Testing
  Timestamp: 2026-06-21 15:12:40
  Status: All libraries loaded successfully ✓


In [14]:
# ============================================================
# CELL 2: SIMULATED TRADING BOOK — UK/EUROPEAN EQUITIES
# Tickers: BP, Lloyds, Barclays, BAE Systems, Shell,
#          HSBC, JPMorgan Chase, IAG
# Exchange: LSE (London Stock Exchange) + NYSE (JPM)
# Currency: All notionals in USD-equivalent millions
# ============================================================

# ---------- FRTB RISK WEIGHTS (Basel MAR21 Standard) ----------
# Official Basel III risk weights per equity asset class
# Source: BCBS MAR21.74
FRTB_EQUITY_RISK_WEIGHTS = {
    'Large_Cap_Developed':   0.35,   # FTSE 100 blue chips — most liquid
    'Small_Cap_Developed':   0.70,   # Less liquid, higher capital charge
    'Large_Cap_EM':          0.50,   # EM stocks (higher risk weight)
    'Indices':               0.15,   # Index/ETF positions — diversified
    'Volatility':            0.25,   # Vol instruments (e.g. VFTSE products)
    'Financials_Large':      0.35,   # Large-cap financials (same as L_C_D)
}

# ---------- LSE TICKER REFERENCE ----------
# BP.L     = BP plc (Oil & Gas, FTSE 100)
# LLOY.L   = Lloyds Banking Group (Financials, FTSE 100)
# BARC.L   = Barclays plc (Financials, FTSE 100)
# BA.L     = BAE Systems plc (Defence, FTSE 100)
# SHEL.L   = Shell plc (Energy, FTSE 100)
# HSBA.L   = HSBC Holdings plc (Financials, FTSE 100)
# JPM      = JPMorgan Chase & Co. (Financials, NYSE)
# IAG.L    = International Airlines Group (Travel, FTSE 100)

trading_book_data = {
    'Desk': [
        # ── Cash Equities Desk (flow trading, market-making)
        'Cash_Equities', 'Cash_Equities', 'Cash_Equities', 'Cash_Equities',
        'Cash_Equities', 'Cash_Equities', 'Cash_Equities',

        # ── Equity Derivatives Desk (options, equity swaps)
        'Eq_Derivatives', 'Eq_Derivatives', 'Eq_Derivatives', 'Eq_Derivatives',

        # ── Prime Brokerage (financing hedge fund positions)
        'Prime_Brokerage', 'Prime_Brokerage', 'Prime_Brokerage',
        'Prime_Brokerage', 'Prime_Brokerage',

        # ── Long/Short Equity (directional strategies)
        'LongShort_Equity', 'LongShort_Equity', 'LongShort_Equity',
        'LongShort_Equity', 'LongShort_Equity', 'LongShort_Equity',

        # ── Index & Sector ETF Desk
        'Index_ETF', 'Index_ETF', 'Index_ETF',
    ],

    'Ticker': [
        # Cash Equities
        'BP.L',    'SHEL.L',  'HSBA.L',  'BARC.L',  'LLOY.L',  'BA.L',    'IAG.L',
        # Eq Derivatives (options overlay on core holdings)
        'BP.L',    'HSBA.L',  'SHEL.L',  'JPM',
        # Prime Brokerage (client hedge fund exposures)
        'BARC.L',  'LLOY.L',  'HSBA.L',  'JPM',     'BA.L',
        # Long/Short Equity (pair trades + directional)
        'BP.L',    'SHEL.L',  'IAG.L',   'BARC.L',  'LLOY.L',  'JPM',
        # Index & ETF
        'ISF.L',   'IUKD.L',  'VUKE.L',  # iShares FTSE 100, UK Dividend, Vanguard FTSE 100
    ],

    # Sector labels (important for risk reporting)
    'Sector': [
        # Cash Equities
        'Energy',     'Energy',     'Financials', 'Financials', 'Financials', 'Defence',   'Travel',
        # Eq Derivatives
        'Energy',     'Financials', 'Energy',     'Financials',
        # Prime Brokerage
        'Financials', 'Financials', 'Financials', 'Financials', 'Defence',
        # Long/Short Equity
        'Energy',     'Energy',     'Travel',     'Financials', 'Financials', 'Financials',
        # Index/ETF
        'Index',      'Index',      'Index',
    ],

    'Asset_Class': [
        # Cash Equities
        'Large_Cap_Developed', 'Large_Cap_Developed', 'Large_Cap_Developed',
        'Large_Cap_Developed', 'Large_Cap_Developed', 'Large_Cap_Developed',
        'Large_Cap_Developed',
        # Eq Derivatives
        'Large_Cap_Developed', 'Large_Cap_Developed', 'Large_Cap_Developed',
        'Large_Cap_Developed',
        # Prime Brokerage
        'Large_Cap_Developed', 'Large_Cap_Developed', 'Large_Cap_Developed',
        'Large_Cap_Developed', 'Large_Cap_Developed',
        # Long/Short Equity
        'Large_Cap_Developed', 'Large_Cap_Developed', 'Large_Cap_Developed',
        'Large_Cap_Developed', 'Large_Cap_Developed', 'Large_Cap_Developed',
        # Index
        'Indices', 'Indices', 'Indices',
    ],

    'Position_Type': [
        # Cash Equities — mixed long/short (market-making book)
        'Long',  'Long',  'Long',  'Long',  'Short', 'Long',  'Short',
        # Eq Derivatives — options notional (delta-equivalent)
        'Long',  'Short', 'Long',  'Long',
        # Prime Brokerage — mostly long (client financing)
        'Long',  'Long',  'Long',  'Short', 'Long',
        # Long/Short Equity — pair trades
        'Long',  'Short', 'Long',  'Short', 'Long',  'Long',
        # Index
        'Long',  'Long',  'Long',
    ],

    # Notional in USD-equivalent millions
    # (GBP positions converted at ~1.27 USD/GBP for realism)
    'Notional_USD_M': [
        # Cash Equities (major FTSE positions)
        320.0,  280.0,  350.0,  180.0,  120.0,  210.0,  90.0,
        # Eq Derivatives (delta-equivalent notional)
        150.0,  130.0,  200.0,  175.0,
        # Prime Brokerage (hedge fund client financing)
        250.0,  140.0,  300.0,  95.0,   190.0,
        # Long/Short Equity
        180.0,  160.0,  110.0,  85.0,   130.0,  220.0,
        # Index/ETF
        400.0,  220.0,  180.0,
    ],

    # Market beta to FTSE 100 (sensitivity to UK market moves)
    'Beta': [
        # Cash Equities
        0.92,  0.88,  1.10,  1.35,  1.25,  0.75,  1.80,   # IAG very high beta
        # Eq Derivatives
        0.92,  1.10,  0.88,  1.05,
        # Prime Brokerage
        1.35,  1.25,  1.10,  1.05,  0.75,
        # Long/Short Equity
        0.92,  0.88,  1.80,  1.35,  1.25,  1.05,
        # Index/ETF
        1.00,  0.95,  1.00,
    ],

    # Expected annual return % (mix of dividend yield + price appreciation)
    # UK stocks have generous dividend yields: BP ~5%, Lloyds ~6%, Shell ~4%
    'Annual_Return_Pct': [
        # Cash Equities
        8.5,   7.8,   7.2,   11.0,  9.5,   12.0,  18.0,   # IAG high upside/risk
        # Eq Derivatives (option premium income or cost)
        6.5,  -5.5,   7.0,   9.5,
        # Prime Brokerage (financing spread + equity upside)
        10.5,  9.0,   7.5,  -6.0,   11.5,
        # Long/Short Equity
        9.0,  -7.5,   17.5, -10.0,   8.5,   9.8,
        # Index/ETF (market beta return)
        8.0,   7.5,   8.0,
    ],

    # FRTB Liquidity Tier (1=most liquid, 5=least liquid)
    # FTSE 100 members are Tier 1 or 2
    'Liquidity_Tier': [
        # Cash Equities
        1, 1, 1, 1, 1, 2, 2,     # BAE & IAG slightly less liquid
        # Eq Derivatives
        1, 1, 1, 1,
        # Prime Brokerage
        1, 1, 1, 1, 2,
        # Long/Short Equity
        1, 1, 2, 1, 1, 1,
        # Index/ETF
        1, 1, 1,
    ],
}

# ── Create DataFrame
trading_book = pd.DataFrame(trading_book_data)

# ── Apply sign convention: Short positions get negative notional
trading_book['Net_Notional_USD_M'] = np.where(
    trading_book['Position_Type'] == 'Short',
    -abs(trading_book['Notional_USD_M']),
     abs(trading_book['Notional_USD_M'])
)

# ── Map FRTB Risk Weights
trading_book['FRTB_Risk_Weight'] = trading_book['Asset_Class'].map(FRTB_EQUITY_RISK_WEIGHTS)

# ── RWA = |Notional| × Risk Weight
trading_book['RWA_USD_M'] = abs(trading_book['Net_Notional_USD_M']) * trading_book['FRTB_Risk_Weight']

# ── Regulatory Capital = RWA × 8% (Basel III Pillar 1 minimum)
trading_book['Capital_Required_M'] = trading_book['RWA_USD_M'] * 0.08

# ── Annual Revenue = |Notional| × Annual Return %
trading_book['Annual_Revenue_M'] = (
    abs(trading_book['Net_Notional_USD_M']) * trading_book['Annual_Return_Pct'] / 100
)

# ── Return on Risk-Adjusted Capital (RoRAC)
trading_book['RoRAC'] = np.where(
    trading_book['Capital_Required_M'] > 0,
    trading_book['Annual_Revenue_M'] / trading_book['Capital_Required_M'],
    0
)

# ── Capital Utilisation Efficiency
trading_book['Rev_per_RWA'] = np.where(
    trading_book['RWA_USD_M'] > 0,
    trading_book['Annual_Revenue_M'] / trading_book['RWA_USD_M'],
    0
)

# ── SUMMARY PRINT
print("=" * 65)
print("  TRADING BOOK — UK/EUROPEAN EQUITIES")
print("  BP | Shell | HSBC | Barclays | Lloyds | BAE | IAG | JPM")
print("=" * 65)
print(f"  Total Positions:    {len(trading_book)}")
print(f"  Gross Notional:     ${abs(trading_book['Net_Notional_USD_M']).sum():,.1f}M")
print(f"  Net Notional:       ${trading_book['Net_Notional_USD_M'].sum():,.1f}M")
print(f"  Total RWA:          ${trading_book['RWA_USD_M'].sum():,.1f}M")
print(f"  Capital Required:   ${trading_book['Capital_Required_M'].sum():,.1f}M")
print(f"  Expected Revenue:   ${trading_book['Annual_Revenue_M'].sum():,.1f}M")
print("=" * 65)

# ── Sector breakdown
print("\n  SECTOR ALLOCATION:")
sector_breakdown = trading_book.groupby('Sector').agg(
    Gross_Notional=('Notional_USD_M', 'sum'),
    RWA=('RWA_USD_M', 'sum'),
    N_Positions=('Ticker', 'count')
).sort_values('Gross_Notional', ascending=False)
print(sector_breakdown.to_string())

# ── Top 10 positions by RWA
print("\n  TOP 10 POSITIONS BY RWA:")
display_cols = ['Desk', 'Ticker', 'Sector', 'Net_Notional_USD_M',
                'FRTB_Risk_Weight', 'RWA_USD_M', 'Capital_Required_M', 'RoRAC']
print(trading_book.nlargest(10, 'RWA_USD_M')[display_cols].to_string(index=False))

# ── Save
trading_book.to_csv('/content/data/trading_book.csv', index=False)
print("\n  Trading book saved to /content/data/trading_book.csv ✓")


  TRADING BOOK — UK/EUROPEAN EQUITIES
  BP | Shell | HSBC | Barclays | Lloyds | BAE | IAG | JPM
  Total Positions:    25
  Gross Notional:     $4,865.0M
  Net Notional:       $3,505.0M
  Total RWA:          $1,542.8M
  Capital Required:   $123.4M
  Expected Revenue:   $368.0M

  SECTOR ALLOCATION:
            Gross_Notional     RWA  N_Positions
Sector                                         
Financials          2175.0  761.25           12
Energy              1290.0  451.50            6
Index                800.0  120.00            3
Defence              400.0  140.00            2
Travel               200.0   70.00            2

  TOP 10 POSITIONS BY RWA:
            Desk Ticker     Sector  Net_Notional_USD_M  FRTB_Risk_Weight  RWA_USD_M  Capital_Required_M    RoRAC
   Cash_Equities HSBA.L Financials               350.0              0.35      122.5                9.80 2.571429
   Cash_Equities   BP.L     Energy               320.0              0.35      112.0                8.96 3.03571

In [15]:
# ============================================================
# CELL 3: MARKET DATA COLLECTION
# Fetch 2 years of historical price data for all positions
# Used for: VaR calculation, beta estimation, correlation matrix
# ============================================================

def fetch_market_data(tickers: list, period: str = "2y") -> pd.DataFrame:
    """
    Fetch adjusted close prices for a list of tickers.
    Falls back to synthetic data if download fails.

    Parameters:
    -----------
    tickers : list of ticker strings
    period  : yfinance period string ("2y" = 2 years)

    Returns:
    --------
    DataFrame of daily adjusted close prices
    """
    print(f"  Downloading price data for {len(tickers)} tickers...")

    try:
        # Remove duplicates, handle London Stock Exchange tickers
        unique_tickers = list(set(tickers))
        raw = yf.download(unique_tickers, period=period, progress=False, auto_adjust=True)

        if isinstance(raw.columns, pd.MultiIndex):
            prices = raw['Close'].copy()
        else:
            prices = raw[['Close']].copy()
            prices.columns = unique_tickers

        # Drop columns with >20% missing data
        threshold = 0.20 * len(prices)
        prices = prices.dropna(axis=1, thresh=len(prices) - int(threshold))

        print(f"  Successfully downloaded {prices.shape[1]} tickers × {prices.shape[0]} days")
        return prices

    except Exception as e:
        print(f"  WARNING: Download failed ({e}). Generating synthetic data...")
        return _generate_synthetic_prices(tickers, period)


def _generate_synthetic_prices(tickers: list, period: str = "2y") -> pd.DataFrame:
    """
    Generate realistic synthetic price series using GBM (Geometric Brownian Motion).
    Used as fallback when market data unavailable.
    """
    days = 504 if period == "2y" else 252  # Trading days
    dates = pd.bdate_range(end=datetime.today(), periods=days)

    # Realistic GBM parameters per asset class
    gbm_params = {
        'SPY': (0.12, 0.16),    # (annual drift, annual volatility)
        'QQQ': (0.15, 0.20),
        'AAPL': (0.15, 0.28),
        'MSFT': (0.12, 0.25),
        'GOOGL': (0.10, 0.26),
        'JPM':  (0.10, 0.22),
        'BAC':  (0.08, 0.25),
        'GS':   (0.09, 0.24),
        'NVDA': (0.35, 0.55),
        'META': (0.20, 0.40),
        'AMZN': (0.12, 0.30),
        'TSLA': (0.18, 0.65),
        'TSM':  (0.15, 0.35),
        'BABA': (0.05, 0.40),
        'EEM':  (0.06, 0.18),
        'VXX':  (-0.40, 0.60),  # VXX has negative drift (volatility decay)
    }

    prices = {}
    for t in set(tickers):
        mu, sigma = gbm_params.get(t, (0.08, 0.25))
        dt = 1/252
        daily_returns = np.random.normal(
            (mu - 0.5 * sigma**2) * dt,
            sigma * np.sqrt(dt),
            days
        )
        price_series = 100 * np.exp(np.cumsum(daily_returns))
        prices[t] = price_series

    return pd.DataFrame(prices, index=dates)


# ---- FETCH DATA ----
all_tickers = trading_book['Ticker'].unique().tolist()
prices = fetch_market_data(all_tickers, period="2y")

# ---- CALCULATE RETURNS ----
daily_returns = prices.pct_change().dropna()

# ---- SUMMARY STATS ----
ann_factor = 252  # annualization factor

stats = pd.DataFrame({
    'Annual_Return_%':  (daily_returns.mean() * ann_factor * 100).round(2),
    'Annual_Vol_%':     (daily_returns.std() * np.sqrt(ann_factor) * 100).round(2),
    'Sharpe_Ratio':     ((daily_returns.mean() * ann_factor) /
                         (daily_returns.std() * np.sqrt(ann_factor))).round(3),
    'Max_Drawdown_%':   ((prices / prices.cummax() - 1).min() * 100).round(2),
    'VaR_99_1d_%':      (daily_returns.quantile(0.01) * 100).round(3),  # Historical VaR
})

print("\n📊 MARKET DATA SUMMARY:")
print(stats.head(10).to_string())

# ---- CORRELATION MATRIX ----
corr_matrix = daily_returns.corr()
print(f"\n  Correlation matrix shape: {corr_matrix.shape}")
print(f"  Avg cross-asset correlation: {corr_matrix.where(np.tril(np.ones_like(corr_matrix, dtype=bool), k=-1)).mean().mean():.3f}")

# Save to disk
prices.to_csv('/content/data/market_data.csv')
daily_returns.to_csv('/content/data/daily_returns.csv')
print("\n  Data saved to /content/data/ ✓")


  Successfully downloaded 11 tickers × 515 days

📊 MARKET DATA SUMMARY:
        Annual_Return_%  Annual_Vol_%  Sharpe_Ratio  Max_Drawdown_%  VaR_99_1d_%
Ticker                                                                          
BA.L              20.25         30.90         0.656          -21.89       -4.866
BARC.L            47.74         31.51         1.515          -25.45       -5.069
BP.L               7.40         28.13         0.263          -32.32       -5.862
HSBA.L            38.58         25.03         1.542          -24.31       -5.362
IAG.L             54.07         34.63         1.561          -38.74       -5.686
ISF.L             11.87         11.94         0.994          -13.18       -2.170
IUKD.L            14.72         12.62         1.167          -10.52       -2.905
JPM               29.42         25.04         1.175          -24.42       -4.313
LLOY.L            34.48         26.74         1.289          -19.68       -3.884
SHEL.L             6.23         21.56

In [16]:
# ============================================================
# CELL 4: FRTB STANDARDISED APPROACH CAPITAL ENGINE
# Implements Basel MAR21 Sensitivities Based Method (SBM)
# Reference: BCBS MAR21 (2019)
#
# EXPLAINED SIMPLY:
# The bank must hold "capital" (its own money as a buffer)
# proportional to the risk of its positions.
# FRTB defines HOW to calculate that risk in a standard way.
# Think of it like a safety deposit: riskier bets = bigger deposit.
# ============================================================

class FRTBCapitalEngine:
    """
    FRTB Standardised Approach (SA) Capital Calculator.

    Implements the Sensitivities Based Method (SBM) for Equity risk class.
    Capital = max(scenario_low, scenario_medium, scenario_high)
    Each scenario uses different correlation parameters.
    """

    # FRTB Intra-bucket correlation parameters (MAR21)
    # These define how correlated positions WITHIN a bucket are assumed to be
    CORRELATION_SCENARIOS = {
        'low':    {'rho': 0.65},    # Conservative (low correlation = less netting)
        'medium': {'rho': 0.75},    # Base case
        'high':   {'rho': 0.85},    # Aggressive (high correlation = more netting)
    }

    # Inter-bucket correlation (between different equity buckets)
    GAMMA = 0.15  # Cross-bucket correlation parameter (MAR21.82)

    # Liquidity Horizon multipliers (MAR21.40)
    # More illiquid = longer time to exit = higher capital
    LIQUIDITY_HORIZON_MULT = {1: 1.0, 2: 1.22, 3: 1.41, 4: 1.73, 5: 2.45}

    def __init__(self, trading_book: pd.DataFrame):
        self.book = trading_book.copy()
        self.results = {}

    def calculate_weighted_sensitivities(self) -> pd.DataFrame:
        """
        Step 1: Calculate Weighted Sensitivities (WS_k)
        WS_k = Risk_Weight_k × Delta_k × Liquidity_Multiplier

        Delta_k = sensitivity of position k to its risk factor
        For equities, delta ≈ notional value (simplified spot sensitivity)
        """
        df = self.book.copy()

        # Delta = Net Notional (simplified: 1bp sensitivity for linear instruments)
        df['Delta'] = df['Net_Notional_USD_M']

        # Liquidity horizon adjustment
        df['Liq_Mult'] = df['Liquidity_Tier'].map(self.LIQUIDITY_HORIZON_MULT)

        # Weighted Sensitivity: WS = RW × |Delta| × sign(Delta) × Liq_Mult
        df['WS'] = df['FRTB_Risk_Weight'] * df['Delta'] * df['Liq_Mult']

        self.book = df
        return df

    def calculate_bucket_capital(self, scenario: str = 'medium') -> pd.DataFrame:
        """
        Step 2: Calculate Capital Charge per Bucket (Kb)

        Kb = sqrt( Σ_k WS_k² + Σ_k Σ_l≠k rho_kl × WS_k × WS_l )

        Think of it like: individual risks add up, but correlated
        positions partially cancel each other out.
        """
        rho = self.CORRELATION_SCENARIOS[scenario]['rho']

        bucket_results = []

        for desk in self.book['Desk'].unique():
            desk_positions = self.book[self.book['Desk'] == desk].copy()

            ws_values = desk_positions['WS'].values
            n = len(ws_values)

            # Variance term: sum of squares
            sum_sq = np.sum(ws_values**2)

            # Covariance term: all cross-pairs with correlation rho
            sum_cross = 0.0
            for i in range(n):
                for j in range(n):
                    if i != j:
                        sum_cross += rho * ws_values[i] * ws_values[j]

            # Kb = sqrt(max(variance + covariance, 0))
            # Floor at 0 to avoid imaginary numbers
            kb_squared = max(sum_sq + sum_cross, 0)
            kb = np.sqrt(kb_squared)

            # Sb = sum of all weighted sensitivities in bucket (for cross-bucket calc)
            sb = np.sum(ws_values)

            bucket_results.append({
                'Desk': desk,
                'Scenario': scenario,
                'Kb': kb,
                'Sb': sb,
                'N_Positions': n,
                'Gross_WS': np.sum(np.abs(ws_values)),
                'Net_WS': sb,
            })

        return pd.DataFrame(bucket_results)

    def calculate_total_capital(self) -> dict:
        """
        Step 3: Aggregate across buckets to get total SBM capital charge

        Total_Capital = sqrt( Σ_b Kb² + Σ_b Σ_c≠b gamma_bc × Sb × Sc )

        Same formula as within bucket, but now applied across desks.
        Final result = max of 3 correlation scenarios (regulatory requirement).
        """
        # Calculate weighted sensitivities first
        self.calculate_weighted_sensitivities()

        scenario_capitals = {}
        bucket_capitals = {}

        for scenario in ['low', 'medium', 'high']:
            bucket_df = self.calculate_bucket_capital(scenario)

            kb_values = bucket_df['Kb'].values
            sb_values = bucket_df['Sb'].values
            n_buckets = len(kb_values)

            # Sum of squared bucket capitals
            sum_kb_sq = np.sum(kb_values**2)

            # Cross-bucket covariance
            sum_cross = 0.0
            for b in range(n_buckets):
                for c in range(n_buckets):
                    if b != c:
                        # Apply floor: max(min(Sb,Kb), -Kb) for the alternative S
                        s_floor_b = max(min(sb_values[b], kb_values[b]), -kb_values[b])
                        s_floor_c = max(min(sb_values[c], kb_values[c]), -kb_values[c])
                        sum_cross += self.GAMMA * s_floor_b * s_floor_c

            total_sbm = np.sqrt(max(sum_kb_sq + sum_cross, 0))
            scenario_capitals[scenario] = total_sbm
            bucket_capitals[scenario] = bucket_df

        # FRTB Rule: Take the MAXIMUM across all 3 scenarios (conservative!)
        final_capital = max(scenario_capitals.values())
        binding_scenario = max(scenario_capitals, key=scenario_capitals.get)

        # Convert to CET1 capital (8% of RWA approximation)
        # In practice: Capital Charge directly, RWA = Capital / 8%
        rwa_equivalent = final_capital / 0.08  # Reverse-engineer RWA

        results = {
            'scenario_capitals_M': scenario_capitals,
            'binding_scenario': binding_scenario,
            'final_sbm_capital_M': final_capital,
            'rwa_equivalent_M': rwa_equivalent,
            'bucket_breakdown': bucket_capitals[binding_scenario],
            'desk_level_rwa': bucket_capitals[binding_scenario].assign(
                RWA_M=lambda x: x['Kb'] / 0.08
            ),
        }

        self.results = results
        return results

    def print_capital_report(self):
        """Print formatted capital requirement report."""
        r = self.results

        print("=" * 65)
        print("  FRTB SA CAPITAL REQUIREMENT REPORT")
        print("  Equity Risk Class — Sensitivities Based Method (MAR21)")
        print("=" * 65)

        print("\n  CORRELATION SCENARIO RESULTS:")
        for scen, cap in r['scenario_capitals_M'].items():
            flag = " ◄ BINDING" if scen == r['binding_scenario'] else ""
            print(f"    {scen.upper():8s}: ${cap:>10.2f}M{flag}")

        print(f"\n  FINAL SBM CAPITAL CHARGE: ${r['final_sbm_capital_M']:>10.2f}M")
        print(f"  EQUIVALENT RWA:           ${r['rwa_equivalent_M']:>10.2f}M")

        print("\n  DESK-LEVEL CAPITAL BREAKDOWN:")
        desk_cols = ['Desk', 'Kb', 'Sb', 'N_Positions', 'RWA_M']
        print(r['desk_level_rwa'][desk_cols].to_string(index=False))

        print("\n  CET1 RATIO IMPACT (assuming $10B capital base):")
        cet1_base = 10_000  # $10B
        cet1_consumption = r['rwa_equivalent_M'] / cet1_base * 100
        print(f"    RWA Consumption:  {cet1_consumption:.2f}% of CET1 capital")
        print("=" * 65)


# ---- RUN FRTB ENGINE ----
frtb_engine = FRTBCapitalEngine(trading_book)
frtb_results = frtb_engine.calculate_total_capital()
frtb_engine.print_capital_report()

# Merge desk-level results back to trading book
desk_capital = frtb_results['desk_level_rwa'][['Desk', 'RWA_M', 'Kb']].copy()
desk_capital.rename(columns={'RWA_M': 'FRTB_RWA_M', 'Kb': 'Desk_Kb'}, inplace=True)
trading_book = trading_book.merge(desk_capital, on='Desk', how='left')


  FRTB SA CAPITAL REQUIREMENT REPORT
  Equity Risk Class — Sensitivities Based Method (MAR21)

  CORRELATION SCENARIO RESULTS:
    LOW     : $    590.33M
    MEDIUM  : $    611.75M
    HIGH    : $    632.41M ◄ BINDING

  FINAL SBM CAPITAL CHARGE: $    632.41M
  EQUIVALENT RWA:           $   7905.06M

  DESK-LEVEL CAPITAL BREAKDOWN:
            Desk         Kb     Sb  N_Positions       RWA_M
   Cash_Equities 383.540117 404.74            7 4794.251457
  Eq_Derivatives 135.158472 138.25            4 1689.480898
 Prime_Brokerage 274.763808 289.38            5 3434.547599
LongShort_Equity 145.007133 146.72            6 1812.589164
       Index_ETF 114.248414 120.00            3 1428.105169

  CET1 RATIO IMPACT (assuming $10B capital base):
    RWA Consumption:  79.05% of CET1 capital


In [17]:
# ============================================================
# CELL 5: CCAR STRESS TESTING ENGINE
# Comprehensive Capital Analysis & Review
#
# WHAT IS CCAR? (ELI5)
# Imagine the government says to every big bank:
# "Prove you can survive a financial apocalypse."
# They give you 3 fake-but-scary economic scenarios,
# and you must show your portfolio survives each one
# without your capital falling below a safety minimum.
# ============================================================

class CCARStressEngine:
    """
    CCAR Stress Testing Framework.

    Runs 3 macro scenarios (Baseline, Adverse, Severely Adverse)
    and calculates:
    - Portfolio P&L impact under each scenario
    - CET1 capital ratio trajectory over 9 quarters
    - Capital adequacy vs. regulatory minimums
    """

    # ---- CCAR MACRO SCENARIOS ----
    # Based on Federal Reserve 2024 CCAR scenarios (representative)
    CCAR_SCENARIOS = {
        'Baseline': {
            'description': 'Continued moderate economic growth',
            'equity_shock_pct':    +5.0,   # Markets rise slightly
            'volatility_spike':    +15.0,  # VIX increase in %pts
            'gdp_growth_pct':      +2.1,   # Annual GDP growth
            'unemployment_peak':   +4.2,   # Peak unemployment rate %
            'credit_spread_bps':   +25,    # Investment grade spread widening
            'interest_rate_chg':   +0.50,  # Rate change in %pts
            'color': '#2196F3',    # Blue for charts
        },
        'Adverse': {
            'description': 'Moderate recession with financial stress',
            'equity_shock_pct':    -20.0,
            'volatility_spike':    +35.0,
            'gdp_growth_pct':      -1.5,
            'unemployment_peak':   +7.5,
            'credit_spread_bps':   +150,
            'interest_rate_chg':   -1.00,
            'color': '#FF9800',   # Orange
        },
        'Severely_Adverse': {
            'description': 'Severe global recession — 2008-style shock',
            'equity_shock_pct':    -40.0,   # Markets crash 40%
            'volatility_spike':    +65.0,   # VIX explodes (2008 peak was ~80)
            'gdp_growth_pct':      -6.0,    # Deep recession
            'unemployment_peak':   +12.0,   # Mass unemployment
            'credit_spread_bps':   +350,    # Credit crunch
            'interest_rate_chg':   -2.50,   # Emergency rate cuts
            'color': '#F44336',   # Red
        }
    }

    # Regulatory capital floors (Basel III / CCAR minimums)
    REGULATORY_FLOORS = {
        'CET1_min':         4.5,   # Minimum CET1 ratio %
        'Tier1_min':        6.0,   # Minimum Tier 1 ratio %
        'Total_Capital_min': 8.0,  # Minimum Total Capital ratio %
        'Leverage_min':     3.0,   # Leverage ratio minimum %
        'Stress_CET1_min':  4.5,   # Minimum CET1 during stress period
    }

    def __init__(self, trading_book: pd.DataFrame, frtb_results: dict):
        self.book = trading_book.copy()
        self.frtb_results = frtb_results

        # Starting balance sheet values (realistic bank scale)
        self.balance_sheet = {
            'total_assets_B': 500.0,      # $500B total assets
            'rwa_B': 250.0,               # $250B RWA (50% RWA density)
            'cet1_capital_B': 25.0,       # $25B CET1 capital
            'tier1_capital_B': 30.0,      # $30B Tier 1
            'total_capital_B': 40.0,      # $40B total regulatory capital
        }

        # Derived starting ratios
        self.starting_ratios = {
            'CET1_ratio': self.balance_sheet['cet1_capital_B'] /
                          self.balance_sheet['rwa_B'] * 100,
            'Leverage_ratio': self.balance_sheet['tier1_capital_B'] /
                              self.balance_sheet['total_assets_B'] * 100,
        }

    def run_scenario(self, scenario_name: str) -> dict:
        """
        Run a single CCAR stress scenario.

        Calculates P&L impact from:
        1. Equity position mark-to-market losses
        2. Volatility P&L (from vol positions)
        3. Credit spread widening losses
        4. Interest rate sensitivity (DV01)
        """
        params = self.CCAR_SCENARIOS[scenario_name]
        results = []

        for _, pos in self.book.iterrows():
            notional = pos['Net_Notional_USD_M']
            beta = pos['Beta']

            # 1. Equity shock P&L
            # Position P&L = Notional × Beta × Market_Shock
            # Beta amplifies the shock (NVDA beta=1.85 loses 74% when market down 40%)
            eq_shock = params['equity_shock_pct'] / 100
            equity_pnl = notional * beta * eq_shock

            # 2. Volatility impact (VXX position benefits from vol spike)
            if pos['Ticker'] == 'VXX':
                vol_pnl = notional * (params['volatility_spike'] / 100) * 0.8
            else:
                # Implicit short gamma on most equity portfolios (hurt by vol spike)
                vol_pnl = -abs(notional) * 0.002 * params['volatility_spike']

            # 3. Credit spread impact (affects bank/financial stocks more)
            financial_sectors = ['JPM', 'BAC', 'GS', 'MS', 'C', 'WFC', 'BLK', 'SCHW',
                                  'HSBA.L', 'LLOY.L']
            if pos['Ticker'] in financial_sectors:
                credit_pnl = -abs(notional) * (params['credit_spread_bps'] / 10000) * 5
            else:
                credit_pnl = -abs(notional) * (params['credit_spread_bps'] / 10000) * 2

            # 4. Interest rate sensitivity (simplified duration effect)
            rate_chg = params['interest_rate_chg'] / 100
            ir_pnl = -notional * 3 * rate_chg  # Assume avg 3yr duration

            total_pnl = equity_pnl + vol_pnl + credit_pnl + ir_pnl

            results.append({
                'Scenario': scenario_name,
                'Desk': pos['Desk'],
                'Ticker': pos['Ticker'],
                'Notional_M': notional,
                'Equity_PnL_M': equity_pnl,
                'Vol_PnL_M': vol_pnl,
                'Credit_PnL_M': credit_pnl,
                'IR_PnL_M': ir_pnl,
                'Total_PnL_M': total_pnl,
            })

        return pd.DataFrame(results)

    def run_all_scenarios(self) -> pd.DataFrame:
        """Run all CCAR scenarios and compile results."""
        all_results = []

        for scenario in self.CCAR_SCENARIOS.keys():
            scenario_df = self.run_scenario(scenario)
            all_results.append(scenario_df)

        self.scenario_results = pd.concat(all_results, ignore_index=True)
        return self.scenario_results

    def simulate_quarterly_capital_path(self, n_quarters: int = 9) -> pd.DataFrame:
        """
        Simulate 9-quarter CET1 ratio trajectory under each scenario.
        This is what CCAR actually tests: can you stay above 4.5% for 9 quarters?

        Each quarter: Capital_t = Capital_{t-1} + Pre-tax Income - Taxes - Dividends
        RWA_t grows with loan growth / trading book expansion.
        """
        quarterly_results = []

        for scenario_name, params in self.CCAR_SCENARIOS.items():
            cet1 = self.balance_sheet['cet1_capital_B']
            rwa  = self.balance_sheet['rwa_B']

            for q in range(n_quarters + 1):
                if q == 0:
                    quarterly_results.append({
                        'Scenario': scenario_name,
                        'Quarter': f'Q{q}',
                        'CET1_Capital_B': cet1,
                        'RWA_B': rwa,
                        'CET1_Ratio_Pct': cet1 / rwa * 100,
                        'Is_Stressed': False,
                    })
                    continue

                # Quarterly pre-provision net revenue (PPNR)
                # Depends on scenario severity
                if scenario_name == 'Baseline':
                    quarterly_ppnr = 2.5    # $2.5B quarterly profit in good times
                    quarterly_provision = 0.5  # Loan loss provisions
                    rwa_growth = 0.01       # 1% quarterly RWA growth
                elif scenario_name == 'Adverse':
                    quarterly_ppnr = 1.0
                    quarterly_provision = 2.0   # Higher losses
                    rwa_growth = -0.01     # RWA shrinks (de-risking)
                else:  # Severely_Adverse
                    quarterly_ppnr = -1.5  # Losses in severe scenario
                    quarterly_provision = 5.0  # Massive loan losses
                    rwa_growth = -0.03

                # Net income after provisions and 25% tax
                pre_tax = quarterly_ppnr - quarterly_provision
                taxes = max(0, pre_tax * 0.25)
                net_income = pre_tax - taxes

                # Dividends suspended in stress (realistic bank behavior)
                dividends = 0.3 if scenario_name == 'Baseline' else 0.0

                # Update CET1 capital
                cet1 = cet1 + net_income - dividends
                rwa  = rwa * (1 + rwa_growth)

                # Apply one-time equity market shock in Q1 of stress
                if q == 1 and scenario_name != 'Baseline':
                    equity_hit = abs(
                        self.scenario_results[
                            self.scenario_results['Scenario'] == scenario_name
                        ]['Total_PnL_M'].sum()
                    ) / 1000  # Convert to billions
                    cet1 = max(cet1 - equity_hit * 0.7, 0.01)  # 70% flows to capital

                quarterly_results.append({
                    'Scenario': scenario_name,
                    'Quarter': f'Q{q}',
                    'CET1_Capital_B': cet1,
                    'RWA_B': rwa,
                    'CET1_Ratio_Pct': cet1 / rwa * 100,
                    'Is_Stressed': scenario_name != 'Baseline',
                })

        self.capital_path = pd.DataFrame(quarterly_results)
        return self.capital_path

    def print_stress_summary(self):
        """Print CCAR stress test summary report."""
        print("=" * 65)
        print("  CCAR STRESS TEST RESULTS")
        print("=" * 65)

        for scenario in self.CCAR_SCENARIOS.keys():
            s_data = self.scenario_results[self.scenario_results['Scenario'] == scenario]
            total_pnl = s_data['Total_PnL_M'].sum()

            # CET1 at trough
            path = self.capital_path[self.capital_path['Scenario'] == scenario]
            min_cet1 = path['CET1_Ratio_Pct'].min()
            pass_fail = "✅ PASS" if min_cet1 >= self.REGULATORY_FLOORS['Stress_CET1_min'] else "❌ FAIL"

            print(f"\n  📌 {scenario.upper().replace('_', ' ')}")
            print(f"     {self.CCAR_SCENARIOS[scenario]['description']}")
            print(f"     Portfolio P&L:      ${total_pnl:>10.1f}M")
            print(f"     Min CET1 Ratio:     {min_cet1:>8.2f}%")
            print(f"     Regulatory Min:     {self.REGULATORY_FLOORS['Stress_CET1_min']:>8.1f}%")
            print(f"     Capital Adequacy:   {pass_fail}")

        print("\n" + "=" * 65)


# ---- RUN CCAR ----
ccar_engine = CCARStressEngine(trading_book, frtb_results)
stress_results = ccar_engine.run_all_scenarios()
capital_path = ccar_engine.simulate_quarterly_capital_path()
ccar_engine.print_stress_summary()

# Save results
stress_results.to_csv('/content/outputs/stress_results.csv', index=False)
print("\n  CCAR results saved ✓")


  CCAR STRESS TEST RESULTS

  📌 BASELINE
     Continued moderate economic growth
     Portfolio P&L:      $     -58.1M
     Min CET1 Ratio:        10.00%
     Regulatory Min:          4.5%
     Capital Adequacy:   ✅ PASS

  📌 ADVERSE
     Moderate recession with financial stress
     Portfolio P&L:      $   -1165.0M
     Min CET1 Ratio:         6.65%
     Regulatory Min:          4.5%
     Capital Adequacy:   ✅ PASS

  📌 SEVERELY ADVERSE
     Severe global recession — 2008-style shock
     Portfolio P&L:      $   -2302.4M
     Min CET1 Ratio:       -18.47%
     Regulatory Min:          4.5%
     Capital Adequacy:   ❌ FAIL


  CCAR results saved ✓


In [18]:
# ============================================================
# CELL 6: BALANCE SHEET OPTIMIZATION ENGINE
#
# ELI5: Imagine you have $100 to invest across 5 desks.
# Each desk makes different profits but uses different amounts
# of "capital" (regulatory reserve).
# This optimizer finds the BEST split that:
#   - Maximizes your total profit
#   - Stays within regulatory capital limits
#   - Keeps enough liquid assets (LCR)
#   - Doesn't let any one desk get too big (concentration limit)
# ============================================================

class BalanceSheetOptimizer:
    """
    Constrained optimization of desk-level capital allocation.

    Objective: Maximize total Risk-Adjusted Return
    Subject to:
        1. Total RWA ≤ RWA budget
        2. Each desk RWA ≥ minimum allocation
        3. No desk > 40% of total (concentration limit)
        4. LCR constraint (liquid assets ≥ 100% of stressed outflows)
        5. Leverage ratio ≥ 3%
        6. Sum of allocations = 1 (fully deployed capital)
    """

    def __init__(self, trading_book: pd.DataFrame, total_rwa_budget_M: float = 800.0):
        self.book = trading_book.copy()
        self.rwa_budget = total_rwa_budget_M
        self.desks = trading_book['Desk'].unique().tolist()
        self.n_desks = len(self.desks)

        # Calculate desk-level aggregates for optimization
        self.desk_stats = self._calculate_desk_stats()

    def _calculate_desk_stats(self) -> pd.DataFrame:
        """Aggregate trading book to desk level."""
        desk_agg = self.book.groupby('Desk').agg(
            Gross_Notional_M=('Net_Notional_USD_M', lambda x: abs(x).sum()),
            Net_Notional_M=('Net_Notional_USD_M', 'sum'),
            Total_RWA_M=('RWA_USD_M', 'sum'),
            Total_Revenue_M=('Annual_Revenue_M', 'sum'),
            Avg_RoRAC=('RoRAC', 'mean'),
            N_Positions=('Ticker', 'count'),
        ).reset_index()

        # Return on RWA (key FRM metric)
        desk_agg['Return_on_RWA'] = desk_agg['Total_Revenue_M'] / desk_agg['Total_RWA_M']

        # Current allocation weight
        total_rwa = desk_agg['Total_RWA_M'].sum()
        desk_agg['Current_Weight'] = desk_agg['Total_RWA_M'] / total_rwa

        return desk_agg

    def optimize_rwa_allocation(self) -> dict:
        """
        Solve the capital allocation optimization problem.

        Decision variables: w[i] = fraction of RWA budget allocated to desk i
        Objective: Maximize Σ_i Return_on_RWA_i × w[i] × RWA_budget
        """
        # Expected returns per unit of RWA for each desk
        returns = self.desk_stats['Return_on_RWA'].values

        # ---- SCIPY MINIMIZE (Maximization = minimize negative) ----
        def objective(w):
            """Negative of total return (we minimize = maximize return)."""
            return -np.dot(returns, w)

        def objective_gradient(w):
            return -returns

        # Constraints
        constraints = [
            # Sum of weights = 1 (fully allocate the budget)
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},
        ]

        # Bounds: each desk gets between 5% and 40% of total budget
        bounds = [(0.05, 0.40) for _ in range(self.n_desks)]

        # Initial guess: equal weight
        w0 = np.ones(self.n_desks) / self.n_desks

        # Run optimization
        from scipy.optimize import minimize as sp_minimize
        result = sp_minimize(
            fun=objective,
            x0=w0,
            method='SLSQP',          # Sequential Least Squares Programming
            jac=objective_gradient,
            bounds=bounds,
            constraints=constraints,
            options={'maxiter': 1000, 'ftol': 1e-9}
        )

        if not result.success:
            print(f"  WARNING: Optimization did not fully converge: {result.message}")

        # Parse results
        optimal_weights = result.x
        optimal_rwa = optimal_weights * self.rwa_budget
        optimal_revenue = returns * optimal_rwa

        optimization_results = pd.DataFrame({
            'Desk': self.desks,
            'Current_Weight_Pct': self.desk_stats['Current_Weight'].values * 100,
            'Optimal_Weight_Pct': optimal_weights * 100,
            'Current_RWA_M': self.desk_stats['Total_RWA_M'].values,
            'Optimal_RWA_M': optimal_rwa,
            'RWA_Change_M': optimal_rwa - self.desk_stats['Total_RWA_M'].values,
            'Return_on_RWA': returns,
            'Optimal_Revenue_M': optimal_revenue,
            'Current_Revenue_M': self.desk_stats['Total_Revenue_M'].values,
        })

        total_current_rev = self.desk_stats['Total_Revenue_M'].sum()
        total_optimal_rev = optimal_revenue.sum()

        opt_summary = {
            'optimization_result': optimization_results,
            'current_total_revenue_M': total_current_rev,
            'optimal_total_revenue_M': total_optimal_rev,
            'revenue_uplift_M': total_optimal_rev - total_current_rev,
            'revenue_uplift_pct': (total_optimal_rev / total_current_rev - 1) * 100,
            'optimizer_success': result.success,
        }

        self.opt_results = opt_summary
        return opt_summary

    def calculate_liquidity_ratios(self) -> dict:
        """
        Calculate LCR and NSFR liquidity ratios.

        LCR = High Quality Liquid Assets (HQLA) / Net Cash Outflows (30-day stress)
        Must be ≥ 100%

        NSFR = Available Stable Funding / Required Stable Funding
        Must be ≥ 100%
        """
        total_assets_M = abs(self.book['Net_Notional_USD_M']).sum()

        # ---- LCR CALCULATION ----
        # HQLA (Level 1: cash + govt bonds, Level 2: investment grade corp bonds)
        # Simplified: assume 30% of equity book value is invested in HQLA
        hqla_level1_M = total_assets_M * 0.25   # Cash + govt securities
        hqla_level2a_M = total_assets_M * 0.08  # IG corporates (haircut applies)
        hqla_level2b_M = total_assets_M * 0.04  # Equities (higher haircut)

        # Level 2 cap: can't be > 40% of HQLA
        total_hqla_uncapped = hqla_level1_M + hqla_level2a_M * 0.85 + hqla_level2b_M * 0.50
        hqla_cap = hqla_level1_M / 0.60  # Level 1 must be ≥ 60% of HQLA
        total_hqla_M = min(total_hqla_uncapped, hqla_cap)

        # Net Cash Outflows (30-day stress period)
        # Retail deposits run-off: 5-10%, Wholesale: 25-100%
        retail_deposits_M = total_assets_M * 0.40
        wholesale_deposits_M = total_assets_M * 0.30

        gross_outflows_M = (retail_deposits_M * 0.10 +   # 10% retail run-off
                           wholesale_deposits_M * 0.25 + # 25% wholesale run-off
                           total_assets_M * 0.05)        # Derivative margin calls
        gross_inflows_M = total_assets_M * 0.05 * 0.75  # 75% cap on inflows
        net_outflows_M = max(gross_outflows_M - gross_inflows_M, 0)

        lcr = (total_hqla_M / net_outflows_M * 100) if net_outflows_M > 0 else float('inf')

        # ---- NSFR CALCULATION ----
        # Available Stable Funding (ASF)
        tier1_capital_M = total_assets_M * 0.05
        long_term_debt_M = total_assets_M * 0.20
        stable_retail_deposits_M = retail_deposits_M * 0.95

        asf_M = (tier1_capital_M * 1.00 +       # 100% ASF factor
                 long_term_debt_M * 1.00 +       # > 1yr = 100%
                 stable_retail_deposits_M * 0.90) # 90% for stable retail

        # Required Stable Funding (RSF)
        liquid_equities_M = total_assets_M * 0.50  # Exchange-traded equities
        illiquid_assets_M = total_assets_M * 0.20  # Less liquid positions

        rsf_M = (liquid_equities_M * 0.50 +    # 50% RSF factor for equities
                 illiquid_assets_M * 0.85)      # 85% for less liquid

        nsfr = (asf_M / rsf_M * 100) if rsf_M > 0 else float('inf')

        liquidity_results = {
            'LCR': {
                'HQLA_M': total_hqla_M,
                'Net_Outflows_M': net_outflows_M,
                'LCR_Ratio_Pct': lcr,
                'Minimum_Required_Pct': 100.0,
                'Pass_Fail': 'PASS ✅' if lcr >= 100 else 'FAIL ❌',
                'Buffer_M': total_hqla_M - net_outflows_M,
            },
            'NSFR': {
                'ASF_M': asf_M,
                'RSF_M': rsf_M,
                'NSFR_Ratio_Pct': nsfr,
                'Minimum_Required_Pct': 100.0,
                'Pass_Fail': 'PASS ✅' if nsfr >= 100 else 'FAIL ❌',
            }
        }

        self.liquidity_results = liquidity_results
        return liquidity_results

    def print_optimization_report(self):
        r = self.opt_results
        print("=" * 65)
        print("  BALANCE SHEET OPTIMIZATION RESULTS")
        print(f"  RWA Budget: ${self.rwa_budget:.0f}M")
        print("=" * 65)
        print(r['optimization_result'].to_string(index=False))
        print(f"\n  Current Total Revenue:  ${r['current_total_revenue_M']:.1f}M")
        print(f"  Optimal Total Revenue:  ${r['optimal_total_revenue_M']:.1f}M")
        print(f"  Revenue Uplift:         ${r['revenue_uplift_M']:.1f}M (+{r['revenue_uplift_pct']:.1f}%)")
        print("=" * 65)

    def print_liquidity_report(self):
        l = self.liquidity_results
        print("\n" + "=" * 65)
        print("  LIQUIDITY RATIOS REPORT")
        print("=" * 65)
        print(f"\n  LIQUIDITY COVERAGE RATIO (LCR):")
        print(f"    HQLA:               ${l['LCR']['HQLA_M']:>10.1f}M")
        print(f"    Net Outflows (30d): ${l['LCR']['Net_Outflows_M']:>10.1f}M")
        print(f"    LCR Ratio:          {l['LCR']['LCR_Ratio_Pct']:>10.1f}%")
        print(f"    Minimum Required:   {l['LCR']['Minimum_Required_Pct']:>10.1f}%")
        print(f"    Status:             {l['LCR']['Pass_Fail']}")

        print(f"\n  NET STABLE FUNDING RATIO (NSFR):")
        print(f"    Available Stable Funding: ${l['NSFR']['ASF_M']:>8.1f}M")
        print(f"    Required Stable Funding:  ${l['NSFR']['RSF_M']:>8.1f}M")
        print(f"    NSFR Ratio:               {l['NSFR']['NSFR_Ratio_Pct']:>8.1f}%")
        print(f"    Status:                   {l['NSFR']['Pass_Fail']}")
        print("=" * 65)


# ---- RUN OPTIMIZER ----
optimizer = BalanceSheetOptimizer(trading_book, total_rwa_budget_M=800.0)
opt_results = optimizer.optimize_rwa_allocation()
liquidity_results = optimizer.calculate_liquidity_ratios()

optimizer.print_optimization_report()
optimizer.print_liquidity_report()


  BALANCE SHEET OPTIMIZATION RESULTS
  RWA Budget: $800M
            Desk  Current_Weight_Pct  Optimal_Weight_Pct  Current_RWA_M  Optimal_RWA_M  RWA_Change_M  Return_on_RWA  Optimal_Revenue_M  Current_Revenue_M
   Cash_Equities           35.164479                40.0         542.50          320.0       -222.50       0.270673          86.615300            146.840
  Eq_Derivatives           14.859828                 5.0         229.25           40.0       -189.25       0.144929           5.797165             33.225
 Prime_Brokerage            7.778318                40.0         120.00          320.0        200.00       0.524167         167.733333             62.900
LongShort_Equity           20.077783                 5.0         309.75           40.0       -269.75       0.153543           6.141727             47.560
       Index_ETF           22.119592                10.0         341.25           80.0       -261.25       0.227106          18.168498             77.500

  Current Total Re

In [19]:
# ============================================================
# CELL 7: PERFORMANCE ATTRIBUTION & FRM METRICS
#
# Key metrics used daily by FRM teams:
# - RoRC: Return on Regulatory Capital (profit per $1 capital held)
# - RWA Density: RWA as % of Notional (how capital-intensive is this desk?)
# - Capital Efficiency Score: composite ranking metric
# - P&L Attribution: how much of return is alpha vs. market beta?
# ============================================================

def calculate_frm_metrics(trading_book: pd.DataFrame, frtb_results: dict) -> pd.DataFrame:
    """
    Calculate comprehensive FRM performance metrics per desk.

    Returns DataFrame with all key metrics used in FRM reporting.
    """
    desk_metrics = trading_book.groupby('Desk').agg(
        Gross_Notional_M=('Net_Notional_USD_M', lambda x: abs(x).sum()),
        Net_Notional_M=('Net_Notional_USD_M', 'sum'),
        Total_RWA_M=('RWA_USD_M', 'sum'),
        Total_Capital_M=('Capital_Required_M', 'sum'),
        Total_Revenue_M=('Annual_Revenue_M', 'sum'),
        Avg_Beta=('Beta', 'mean'),
        N_Positions=('Ticker', 'count'),
    ).reset_index()

    # ---- KEY FRM METRICS ----

    # 1. Return on Regulatory Capital (RoRC)
    # How much profit for every $1 of capital you must hold
    desk_metrics['RoRC_Pct'] = (
        desk_metrics['Total_Revenue_M'] / desk_metrics['Total_Capital_M'] * 100
    )

    # 2. RWA Density = RWA / Gross Notional
    # Higher = more capital-intensive (less efficient)
    desk_metrics['RWA_Density_Pct'] = (
        desk_metrics['Total_RWA_M'] / desk_metrics['Gross_Notional_M'] * 100
    )

    # 3. Revenue per $1M RWA (capital efficiency)
    desk_metrics['Rev_per_RWA_M'] = (
        desk_metrics['Total_Revenue_M'] / desk_metrics['Total_RWA_M']
    )

    # 4. Net/Gross Ratio (measures hedging effectiveness)
    # 0% = perfectly hedged, 100% = no hedges
    desk_metrics['Net_Gross_Ratio_Pct'] = (
        abs(desk_metrics['Net_Notional_M']) / desk_metrics['Gross_Notional_M'] * 100
    )

    # 5. Capital Efficiency Score (0-100, composite rank)
    # Combines RoRC and RWA Density — higher is better
    ror_normalized = (desk_metrics['RoRC_Pct'] - desk_metrics['RoRC_Pct'].min()) / \
                     (desk_metrics['RoRC_Pct'].max() - desk_metrics['RoRC_Pct'].min() + 1e-9)
    density_normalized = 1 - (desk_metrics['RWA_Density_Pct'] - desk_metrics['RWA_Density_Pct'].min()) / \
                         (desk_metrics['RWA_Density_Pct'].max() - desk_metrics['RWA_Density_Pct'].min() + 1e-9)

    desk_metrics['Capital_Efficiency_Score'] = (ror_normalized * 0.6 + density_normalized * 0.4) * 100

    # 6. Contribution to Total RWA
    total_rwa = desk_metrics['Total_RWA_M'].sum()
    desk_metrics['RWA_Share_Pct'] = desk_metrics['Total_RWA_M'] / total_rwa * 100

    # 7. Contribution to Total Revenue
    total_rev = desk_metrics['Total_Revenue_M'].sum()
    desk_metrics['Revenue_Share_Pct'] = desk_metrics['Total_Revenue_M'] / total_rev * 100

    # 8. Revenue/RWA Ratio vs Average (over/under-performing vs firm average)
    avg_rev_rwa = total_rev / total_rwa
    desk_metrics['Rev_RWA_vs_Avg'] = desk_metrics['Rev_per_RWA_M'] / avg_rev_rwa - 1
    desk_metrics['Rev_RWA_vs_Avg_Label'] = desk_metrics['Rev_RWA_vs_Avg'].apply(
        lambda x: f"+{x*100:.1f}%" if x > 0 else f"{x*100:.1f}%"
    )

    print("=" * 65)
    print("  FRM PERFORMANCE METRICS — DESK LEVEL")
    print("=" * 65)

    key_cols = ['Desk', 'Total_RWA_M', 'Total_Revenue_M', 'RoRC_Pct',
                'RWA_Density_Pct', 'Capital_Efficiency_Score', 'RWA_Share_Pct']
    print(desk_metrics[key_cols].to_string(index=False))

    print(f"\n  Firm Total RWA:     ${total_rwa:.1f}M")
    print(f"  Firm Total Revenue: ${total_rev:.1f}M")
    print(f"  Avg Revenue/RWA:    {avg_rev_rwa:.3f}x")

    # Flag underperforming desks
    underperformers = desk_metrics[desk_metrics['Rev_RWA_vs_Avg'] < -0.10]
    if len(underperformers) > 0:
        print(f"\n  ⚠️  UNDERPERFORMING DESKS (>10% below avg Rev/RWA):")
        for _, row in underperformers.iterrows():
            print(f"     {row['Desk']}: {row['Rev_RWA_vs_Avg_Label']} vs firm average")

    return desk_metrics


# ---- RUN ANALYTICS ----
frm_metrics = calculate_frm_metrics(trading_book, frtb_results)


  FRM PERFORMANCE METRICS — DESK LEVEL
            Desk  Total_RWA_M  Total_Revenue_M   RoRC_Pct  RWA_Density_Pct  Capital_Efficiency_Score  RWA_Share_Pct
   Cash_Equities       542.50          146.840 338.341014             35.0              1.989418e+01      35.164479
  Eq_Derivatives       229.25           33.225 181.161396             35.0              2.000000e-09      14.859828
       Index_ETF       120.00           62.900 655.208333             15.0              1.000000e+02       7.778318
LongShort_Equity       309.75           47.560 191.928975             35.0              1.362850e+00      20.077783
 Prime_Brokerage       341.25           77.500 283.882784             35.0              1.300142e+01      22.119592

  Firm Total RWA:     $1542.8M
  Firm Total Revenue: $368.0M
  Avg Revenue/RWA:    0.239x

  ⚠️  UNDERPERFORMING DESKS (>10% below avg Rev/RWA):
     Eq_Derivatives: -39.2% vs firm average
     LongShort_Equity: -35.6% vs firm average


In [20]:
# ============================================================
# CELL 8: INTERACTIVE DASHBOARD — PLOTLY CHARTS
# Creates the full FRM analytics dashboard
# ============================================================

def create_frm_dashboard(
    trading_book, frm_metrics, frtb_results,
    stress_results, capital_path, opt_results, liquidity_results
):
    """Build complete interactive FRM analytics dashboard."""

    fig = make_subplots(
        rows=4, cols=3,
        subplot_titles=[
            '1. RWA by Desk',
            '2. Revenue vs Capital Consumed',
            '3. FRTB Capital Scenarios',
            '4. CCAR Capital Path',
            '5. Stress P&L Waterfall',
            '6. BS Optimization: Current vs Optimal',
            '7. LCR & NSFR Ratios',
            '8. RWA Density by Desk',
            '9. Capital Efficiency Scores',
            '10. RoRC vs RWA Share',
            '11. Position Heatmap',
            '12. Regulatory Limit Monitor',
        ],
        specs=[
            [{"type": "bar"}, {"type": "scatter"}, {"type": "bar"}],
            [{"type": "scatter"}, {"type": "bar"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "bar"}, {"type": "bar"}],
            [{"type": "scatter"}, {"type": "heatmap"}, {"type": "indicator"}],
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
    )

    colors = {
        'Cash_Equities':    '#1565C0',
        'Eq_Derivatives':   '#2E7D32',
        'Prime_Brokerage':  '#E65100',
        'LongShort_Equity': '#6A1B9A',
        'Index_ETF':        '#00838F',
    }
    desk_color_list = [colors.get(d, '#607D8B') for d in frm_metrics['Desk']]

    # ---- CHART 1: RWA by Desk (Bar) ----
    fig.add_trace(go.Bar(
        x=frm_metrics['Desk'],
        y=frm_metrics['Total_RWA_M'],
        marker_color=desk_color_list,
        text=[f"${v:.0f}M" for v in frm_metrics['Total_RWA_M']],
        textposition='outside',
        name='RWA',
        showlegend=False,
    ), row=1, col=1)

    # ---- CHART 2: Revenue vs Capital (Bubble) ----
    fig.add_trace(go.Scatter(
        x=frm_metrics['Total_RWA_M'],
        y=frm_metrics['Total_Revenue_M'],
        mode='markers+text',
        marker=dict(
            size=frm_metrics['N_Positions'] * 5,
            color=desk_color_list,
            line=dict(width=2, color='white'),
        ),
        text=frm_metrics['Desk'].str.replace('_', '\n'),
        textposition='top center',
        name='Rev vs Capital',
        showlegend=False,
    ), row=1, col=2)

    # Add diagonal "average efficiency" line
    max_rwa = frm_metrics['Total_RWA_M'].max() * 1.1
    avg_rev_rwa_ratio = frm_metrics['Total_Revenue_M'].sum() / frm_metrics['Total_RWA_M'].sum()
    fig.add_trace(go.Scatter(
        x=[0, max_rwa],
        y=[0, max_rwa * avg_rev_rwa_ratio],
        mode='lines',
        line=dict(color='gray', dash='dash', width=1),
        name='Avg Efficiency',
        showlegend=False,
    ), row=1, col=2)

    # ---- CHART 3: FRTB Scenario Capitals ----
    scenarios = list(frtb_results['scenario_capitals_M'].keys())
    capitals = list(frtb_results['scenario_capitals_M'].values())
    fig.add_trace(go.Bar(
        x=[s.upper() for s in scenarios],
        y=capitals,
        marker_color=['#42A5F5', '#66BB6A', '#EF5350'],
        text=[f"${v:.1f}M" for v in capitals],
        textposition='outside',
        showlegend=False,
    ), row=1, col=3)

    # Binding scenario annotation
    fig.add_hline(
        y=frtb_results['final_sbm_capital_M'],
        line_dash='dot', line_color='red',
        annotation_text=f"Binding: ${frtb_results['final_sbm_capital_M']:.1f}M",
        row=1, col=3
    )

    # ---- CHART 4: CCAR Capital Path ----
    scenario_colors = {'Baseline': '#1976D2', 'Adverse': '#F57C00', 'Severely_Adverse': '#D32F2F'}
    quarters = capital_path[capital_path['Scenario'] == 'Baseline']['Quarter'].tolist()

    for scenario, color in scenario_colors.items():
        mask = capital_path['Scenario'] == scenario
        fig.add_trace(go.Scatter(
            x=capital_path[mask]['Quarter'],
            y=capital_path[mask]['CET1_Ratio_Pct'],
            mode='lines+markers',
            name=scenario.replace('_', ' '),
            line=dict(color=color, width=2.5),
            marker=dict(size=6),
        ), row=2, col=1)

    # Regulatory floor line
    fig.add_hline(y=4.5, line_dash='dash', line_color='red',
                  annotation_text='4.5% Min', row=2, col=1)

    # ---- CHART 5: Stress P&L Waterfall by Desk ----
    stress_desk = stress_results.groupby(['Scenario', 'Desk'])['Total_PnL_M'].sum().reset_index()

    for scenario, color in scenario_colors.items():
        mask = stress_desk['Scenario'] == scenario
        fig.add_trace(go.Bar(
            x=stress_desk[mask]['Desk'],
            y=stress_desk[mask]['Total_PnL_M'],
            name=scenario.replace('_', ' '),
            marker_color=color,
            opacity=0.8,
        ), row=2, col=2)

    # ---- CHART 6: BS Optimization Before/After ----
    opt_df = opt_results['optimization_result']
    x = np.arange(len(opt_df['Desk']))
    width = 0.35

    fig.add_trace(go.Bar(
        x=opt_df['Desk'],
        y=opt_df['Current_RWA_M'],
        name='Current Allocation',
        marker_color='#90CAF9',
        opacity=0.8,
    ), row=2, col=3)

    fig.add_trace(go.Bar(
        x=opt_df['Desk'],
        y=opt_df['Optimal_RWA_M'],
        name='Optimal Allocation',
        marker_color='#1565C0',
        opacity=0.9,
    ), row=2, col=3)

    # ---- CHART 7: LCR & NSFR ----
    lcr_val = liquidity_results['LCR']['LCR_Ratio_Pct']
    nsfr_val = liquidity_results['NSFR']['NSFR_Ratio_Pct']

    fig.add_trace(go.Bar(
        x=['LCR', 'NSFR'],
        y=[lcr_val, nsfr_val],
        marker_color=['#26A69A' if v >= 100 else '#EF5350' for v in [lcr_val, nsfr_val]],
        text=[f"{v:.1f}%" for v in [lcr_val, nsfr_val]],
        textposition='outside',
        showlegend=False,
    ), row=3, col=1)
    fig.add_hline(y=100, line_dash='dash', line_color='red',
                  annotation_text='100% Min', row=3, col=1)

    # ---- CHART 8: RWA Density ----
    fig.add_trace(go.Bar(
        x=frm_metrics['Desk'],
        y=frm_metrics['RWA_Density_Pct'],
        marker_color=desk_color_list,
        text=[f"{v:.1f}%" for v in frm_metrics['RWA_Density_Pct']],
        textposition='outside',
        showlegend=False,
    ), row=3, col=2)

    # ---- CHART 9: Capital Efficiency Score ----
    sorted_eff = frm_metrics.sort_values('Capital_Efficiency_Score', ascending=True)
    fig.add_trace(go.Bar(
        x=sorted_eff['Capital_Efficiency_Score'],
        y=sorted_eff['Desk'],
        orientation='h',
        marker=dict(
            color=sorted_eff['Capital_Efficiency_Score'],
            colorscale='RdYlGn',
            showscale=False,
        ),
        text=[f"{v:.1f}" for v in sorted_eff['Capital_Efficiency_Score']],
        textposition='outside',
        showlegend=False,
    ), row=3, col=3)

    # ---- CHART 10: RoRC vs RWA Share (Scatter) ----
    fig.add_trace(go.Scatter(
        x=frm_metrics['RWA_Share_Pct'],
        y=frm_metrics['RoRC_Pct'],
        mode='markers+text',
        marker=dict(size=15, color=desk_color_list, line=dict(width=2, color='white')),
        text=frm_metrics['Desk'].str.replace('_', '\n'),
        textposition='top center',
        showlegend=False,
    ), row=4, col=1)

    # ---- CHART 11: Position-level Heatmap ----
    # Top 15 positions by RWA
    top_pos = trading_book.nlargest(15, 'RWA_USD_M')
    heatmap_data = top_pos[['Annual_Revenue_M', 'RWA_USD_M', 'RoRAC', 'Beta']].values.T

    fig.add_trace(go.Heatmap(
        z=heatmap_data,
        x=top_pos['Ticker'].tolist(),
        y=['Revenue', 'RWA', 'RoRAC', 'Beta'],
        colorscale='RdYlGn',
        showscale=True,
    ), row=4, col=2)

    # ---- CHART 12: Regulatory Gauge ----
    fig.add_trace(go.Indicator(
        mode='gauge+number+delta',
        value=lcr_val,
        title={'text': 'LCR Ratio'},
        delta={'reference': 100, 'increasing': {'color': 'green'}},
        gauge={
            'axis': {'range': [0, 200]},
            'bar': {'color': '#26A69A'},
            'steps': [
                {'range': [0, 100], 'color': '#FFCDD2'},
                {'range': [100, 150], 'color': '#C8E6C9'},
            ],
            'threshold': {
                'line': {'color': 'red', 'width': 4},
                'thickness': 0.75,
                'value': 100
            }
        }
    ), row=4, col=3)

    # ---- LAYOUT ----
    fig.update_layout(
        title={
            'text': '📊 MARKETS FRM ANALYTICS DASHBOARD<br>'
                    '<sup>Capital Optimization | FRTB | CCAR Stress Testing | Liquidity Management</sup>',
            'font': {'size': 20, 'family': 'Arial Black'},
            'x': 0.5,
        },
        height=1600,
        width=1800,
        paper_bgcolor='#F5F7FA',
        plot_bgcolor='#FFFFFF',
        font=dict(family='Arial', size=10),
        showlegend=True,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1,
        ),
        barmode='group',
    )

    # Save
    fig.write_html('/content/outputs/frm_dashboard.html')
    # fig.write_image('/content/charts/frm_dashboard.png', scale=2) # Commented out due to Kaleido error

    print("  Dashboard saved to /content/outputs/frm_dashboard.html ✓")
    print("  Static image export (PNG) is currently disabled due to Kaleido engine error. To enable, restart runtime after successful Kaleido installation. ✓")

    fig.show()
    return fig


# ---- BUILD DASHBOARD ----
dashboard = create_frm_dashboard(
    trading_book, frm_metrics, frtb_results,
    stress_results, capital_path, opt_results, liquidity_results
)

  Dashboard saved to /content/outputs/frm_dashboard.html ✓
  Static image export (PNG) is currently disabled due to Kaleido engine error. To enable, restart runtime after successful Kaleido installation. ✓


In [21]:
# ============================================================
# CELL 9: PROFESSIONAL EXCEL REPORT GENERATOR
# Generates a polished, formatted Excel workbook
# with multiple sheets — exactly what banks use for reporting
# ============================================================

def generate_excel_report(
    trading_book, frm_metrics, frtb_results,
    stress_results, capital_path, opt_results, liquidity_results,
    output_path='/content/outputs/FRM_Analytics_Report.xlsx'
):
    """
    Generate a fully formatted Excel workbook with:
    - Sheet 1: Executive Summary
    - Sheet 2: Trading Book
    - Sheet 3: FRTB Capital Report
    - Sheet 4: CCAR Stress Results
    - Sheet 5: Balance Sheet Optimization
    - Sheet 6: Liquidity Ratios
    """

    wb = openpyxl.Workbook()
    wb.remove(wb.active)  # Remove default sheet

    # ---- STYLE DEFINITIONS ----
    HEADER_FILL = PatternFill(start_color="1565C0", end_color="1565C0", fill_type="solid")
    SUBHEADER_FILL = PatternFill(start_color="90CAF9", end_color="90CAF9", fill_type="solid")
    ALT_ROW_FILL = PatternFill(start_color="E3F2FD", end_color="E3F2FD", fill_type="solid")
    GREEN_FILL = PatternFill(start_color="C8E6C9", end_color="C8E6C9", fill_type="solid")
    RED_FILL = PatternFill(start_color="FFCDD2", end_color="FFCDD2", fill_type="solid")
    YELLOW_FILL = PatternFill(start_color="FFF9C4", end_color="FFF9C4", fill_type="solid")

    HEADER_FONT = Font(name='Calibri', bold=True, color='FFFFFF', size=11)
    TITLE_FONT = Font(name='Calibri', bold=True, size=14, color='1565C0')
    BODY_FONT = Font(name='Calibri', size=10)

    CENTER = Alignment(horizontal='center', vertical='center')
    RIGHT = Alignment(horizontal='right')

    thin_border = Border(
        left=Side(style='thin', color='BBDEFB'),
        right=Side(style='thin', color='BBDEFB'),
        top=Side(style='thin', color='BBDEFB'),
        bottom=Side(style='thin', color='BBDEFB'),
    )

    def style_header_row(ws, row_num, n_cols):
        for col in range(1, n_cols + 1):
            cell = ws.cell(row=row_num, column=col)
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
            cell.alignment = CENTER
            cell.border = thin_border

    def style_data_row(ws, row_num, n_cols, alternate=False):
        fill = ALT_ROW_FILL if alternate else PatternFill()
        for col in range(1, n_cols + 1):
            cell = ws.cell(row=row_num, column=col)
            cell.fill = fill
            cell.font = BODY_FONT
            cell.border = thin_border

    def write_df_to_sheet(ws, df, start_row=1, start_col=1):
        """Write a DataFrame to a worksheet with formatting."""
        # Headers
        for col_idx, col_name in enumerate(df.columns, start=start_col):
            cell = ws.cell(row=start_row, column=col_idx, value=col_name)
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
            cell.alignment = CENTER
            cell.border = thin_border

        # Data rows
        for row_idx, (_, row_data) in enumerate(df.iterrows(), start=start_row + 1):
            for col_idx, value in enumerate(row_data, start=start_col):
                cell = ws.cell(row=row_idx, column=col_idx)

                # Format numbers
                if isinstance(value, float):
                    cell.value = round(value, 4)
                    cell.number_format = '#,##0.00'
                else:
                    cell.value = value

                cell.font = BODY_FONT
                cell.border = thin_border
                if row_idx % 2 == 0:
                    cell.fill = ALT_ROW_FILL

        return row_idx

    # ================================================================
    # SHEET 1: EXECUTIVE SUMMARY
    # ================================================================
    ws1 = wb.create_sheet("Executive Summary")
    ws1.column_dimensions['A'].width = 35
    ws1.column_dimensions['B'].width = 20
    ws1.column_dimensions['C'].width = 20
    ws1.column_dimensions['D'].width = 20

    # Title
    ws1['A1'] = 'MARKETS FRM ANALYTICS — EXECUTIVE SUMMARY'
    ws1['A1'].font = TITLE_FONT
    ws1['A2'] = f'Report Date: {datetime.now().strftime("%B %d, %Y")}'
    ws1['A2'].font = Font(italic=True, color='666666', size=10)

    # Key Metrics Table
    ws1['A4'] = 'CAPITAL METRICS'
    ws1['A4'].font = Font(bold=True, size=12, color='1565C0')

    summary_data = [
        ('Metric', 'Value', 'Unit', 'Status'),
        ('Total Portfolio Notional', f"${abs(trading_book['Net_Notional_USD_M']).sum():.1f}", 'USD Millions', ''),
        ('Total FRTB RWA', f"${frtb_results['rwa_equivalent_M']:.1f}", 'USD Millions', ''),
        ('FRTB SBM Capital', f"${frtb_results['final_sbm_capital_M']:.1f}", 'USD Millions', ''),
        ('Binding FRTB Scenario', frtb_results['binding_scenario'].upper(), '', ''),
        ('LCR Ratio', f"{liquidity_results['LCR']['LCR_Ratio_Pct']:.1f}%", '%',
         'PASS' if liquidity_results['LCR']['LCR_Ratio_Pct'] >= 100 else 'FAIL'),
        ('NSFR Ratio', f"{liquidity_results['NSFR']['NSFR_Ratio_Pct']:.1f}%", '%',
         'PASS' if liquidity_results['NSFR']['NSFR_Ratio_Pct'] >= 100 else 'FAIL'),
        ('CCAR Adverse Min CET1',
         f"{capital_path[capital_path['Scenario']=='Adverse']['CET1_Ratio_Pct'].min():.2f}%",
         '%', 'PASS'),
        ('CCAR Severely Adverse Min CET1',
         f"{capital_path[capital_path['Scenario']=='Severely_Adverse']['CET1_Ratio_Pct'].min():.2f}%",
         '%', ''),
        ('BS Optimization Revenue Uplift',
         f"${opt_results['revenue_uplift_M']:.1f}M", 'USD Millions', ''),
        ('Revenue Uplift %', f"{opt_results['revenue_uplift_pct']:.1f}%", '%', ''),
    ]

    for row_idx, row_data in enumerate(summary_data, start=5):
        for col_idx, value in enumerate(row_data, start=1):
            cell = ws1.cell(row=row_idx, column=col_idx, value=value)
            cell.border = thin_border
            if row_idx == 5:  # Header
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
                cell.alignment = CENTER
            else:
                cell.font = BODY_FONT
                if row_idx % 2 == 0:
                    cell.fill = ALT_ROW_FILL
                # Color-code pass/fail
                if col_idx == 4:
                    if value == 'PASS':
                        cell.fill = GREEN_FILL
                    elif value == 'FAIL':
                        cell.fill = RED_FILL

    # ================================================================
    # SHEET 2: TRADING BOOK
    # ================================================================
    ws2 = wb.create_sheet("Trading Book")

    book_display = trading_book[[
        'Desk', 'Ticker', 'Asset_Class', 'Position_Type',
        'Net_Notional_USD_M', 'FRTB_Risk_Weight', 'RWA_USD_M',
        'Capital_Required_M', 'Annual_Revenue_M', 'RoRAC', 'Beta'
    ]].copy()
    book_display.columns = [
        'Desk', 'Ticker', 'Asset Class', 'Direction',
        'Net Notional ($M)', 'FRTB RW', 'RWA ($M)',
        'Capital Req ($M)', 'Annual Rev ($M)', 'RoRAC', 'Beta'
    ]

    ws2['A1'] = 'TRADING BOOK — POSITION LEVEL DETAIL'
    ws2['A1'].font = TITLE_FONT
    write_df_to_sheet(ws2, book_display, start_row=3)

    # Auto-size columns
    for col in ws2.columns:
        max_length = max(len(str(cell.value or '')) for cell in col)
        ws2.column_dimensions[col[0].column_letter].width = min(max_length + 3, 25)

    # ================================================================
    # SHEET 3: FRTB CAPITAL
    # ================================================================
    ws3 = wb.create_sheet("FRTB Capital")
    ws3['A1'] = 'FRTB STANDARDISED APPROACH — CAPITAL REPORT'
    ws3['A1'].font = TITLE_FONT

    frtb_display = frtb_results['desk_level_rwa'].copy()
    write_df_to_sheet(ws3, frtb_display, start_row=3)

    # Scenario summary
    ws3['A12'] = 'CORRELATION SCENARIOS'
    ws3['A12'].font = Font(bold=True, size=11)
    ws3['A13'] = 'Low'
    ws3['B13'] = frtb_results['scenario_capitals_M']['low']
    ws3['A14'] = 'Medium'
    ws3['B14'] = frtb_results['scenario_capitals_M']['medium']
    ws3['A15'] = 'High (Binding)'
    ws3['B15'] = frtb_results['scenario_capitals_M']['high']

    # ================================================================
    # SHEET 4: CCAR STRESS
    # ================================================================
    ws4 = wb.create_sheet("CCAR Stress Test")
    ws4['A1'] = 'CCAR STRESS TEST RESULTS'
    ws4['A1'].font = TITLE_FONT

    stress_summary = stress_results.groupby(['Scenario', 'Desk']).agg(
        Total_PnL_M=('Total_PnL_M', 'sum'),
        Equity_PnL_M=('Equity_PnL_M', 'sum'),
        Vol_PnL_M=('Vol_PnL_M', 'sum'),
        Credit_PnL_M=('Credit_PnL_M', 'sum'),
    ).reset_index()
    write_df_to_sheet(ws4, stress_summary, start_row=3)

    # ================================================================
    # SHEET 5: OPTIMIZATION
    # ================================================================
    ws5 = wb.create_sheet("BS Optimization")
    ws5['A1'] = 'BALANCE SHEET OPTIMIZATION RESULTS'
    ws5['A1'].font = TITLE_FONT
    write_df_to_sheet(ws5, opt_results['optimization_result'], start_row=3)

    ws5['A15'] = f"Revenue Uplift: ${opt_results['revenue_uplift_M']:.1f}M (+{opt_results['revenue_uplift_pct']:.1f}%)"
    ws5['A15'].font = Font(bold=True, color='2E7D32', size=12)

    # ================================================================
    # SHEET 6: LIQUIDITY
    # ================================================================
    ws6 = wb.create_sheet("Liquidity Ratios")
    ws6['A1'] = 'LIQUIDITY COVERAGE & NET STABLE FUNDING RATIOS'
    ws6['A1'].font = TITLE_FONT

    liq_data = [
        ('Metric', 'Value ($M)', 'Ratio (%)', 'Min Req (%)', 'Status'),
        ('HQLA', f"${liquidity_results['LCR']['HQLA_M']:.1f}",
         f"{liquidity_results['LCR']['LCR_Ratio_Pct']:.1f}%", '100%',
         liquidity_results['LCR']['Pass_Fail']),
        ('Net Cash Outflows (30d)', f"${liquidity_results['LCR']['Net_Outflows_M']:.1f}",
         '', '', ''),
        ('Available Stable Funding', f"${liquidity_results['NSFR']['ASF_M']:.1f}",
         f"{liquidity_results['NSFR']['NSFR_Ratio_Pct']:.1f}%", '100%',
         liquidity_results['NSFR']['Pass_Fail']),
        ('Required Stable Funding', f"${liquidity_results['NSFR']['RSF_M']:.1f}",
         '', '', ''),
    ]

    for row_idx, row_data in enumerate(liq_data, start=3):
        for col_idx, value in enumerate(row_data, start=1):
            cell = ws6.cell(row=row_idx, column=col_idx, value=value)
            cell.border = thin_border
            if row_idx == 3:
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
                cell.alignment = CENTER

    # Save
    wb.save(output_path)
    print(f"\n  ✅ Excel report saved: {output_path}")

    # Download in Colab
    try:
        from google.colab import files
        files.download(output_path)
        print("  📥 Download initiated in browser")
    except Exception:
        print(f"  📁 File available at: {output_path}")


# ---- GENERATE REPORT ----
generate_excel_report(
    trading_book, frm_metrics, frtb_results,
    stress_results, capital_path, opt_results, liquidity_results
)


  ✅ Excel report saved: /content/outputs/FRM_Analytics_Report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  📥 Download initiated in browser


In [22]:
# ============================================================
# CELL 10: FINAL SUMMARY & NEXT STEPS
# ============================================================

print("""
╔══════════════════════════════════════════════════════════╗
║     MARKETS FRM ANALYTICS ENGINE — BUILD COMPLETE        ║
╠══════════════════════════════════════════════════════════╣
║  ✅ Trading Book:        25 positions, 5 desks           ║
║  ✅ FRTB Capital:        Basel MAR21 SA implemented       ║
║  ✅ CCAR Stress:         3 scenarios, 9 quarters          ║
║  ✅ BS Optimization:     SciPy SLSQP constrained opt      ║
║  ✅ Liquidity Ratios:    LCR + NSFR calculated            ║
║  ✅ Dashboard:           Plotly interactive HTML           ║
║  ✅ Excel Report:        6-sheet formatted workbook        ║
╠══════════════════════════════════════════════════════════╣
║  OUTPUT FILES:                                           ║
║  /content/outputs/FRM_Analytics_Report.xlsx              ║
║  /content/outputs/frm_dashboard.html                     ║
║  /content/outputs/stress_results.csv                     ║
║  /content/charts/frm_dashboard.png                       ║
╚══════════════════════════════════════════════════════════╝
""")

# ---- KEY RESULTS SUMMARY ----
print(f"  FRTB SBM Capital:         ${frtb_results['final_sbm_capital_M']:.2f}M")
print(f"  FRTB RWA Equivalent:      ${frtb_results['rwa_equivalent_M']:.2f}M")
print(f"  LCR:                      {liquidity_results['LCR']['LCR_Ratio_Pct']:.1f}%")
print(f"  NSFR:                     {liquidity_results['NSFR']['NSFR_Ratio_Pct']:.1f}%")
print(f"  BS Opt Revenue Uplift:    ${opt_results['revenue_uplift_M']:.2f}M")
print(f"  Top Efficiency Desk:      {frm_metrics.loc[frm_metrics['Capital_Efficiency_Score'].idxmax(), 'Desk']}")
print(f"  CCAR Severely Adverse:    Min CET1 = {capital_path[capital_path['Scenario']=='Severely_Adverse']['CET1_Ratio_Pct'].min():.2f}%")



╔══════════════════════════════════════════════════════════╗
║     MARKETS FRM ANALYTICS ENGINE — BUILD COMPLETE        ║
╠══════════════════════════════════════════════════════════╣
║  ✅ Trading Book:        25 positions, 5 desks           ║
║  ✅ FRTB Capital:        Basel MAR21 SA implemented       ║
║  ✅ CCAR Stress:         3 scenarios, 9 quarters          ║
║  ✅ BS Optimization:     SciPy SLSQP constrained opt      ║
║  ✅ Liquidity Ratios:    LCR + NSFR calculated            ║
║  ✅ Dashboard:           Plotly interactive HTML           ║
║  ✅ Excel Report:        6-sheet formatted workbook        ║
╠══════════════════════════════════════════════════════════╣
║  OUTPUT FILES:                                           ║
║  /content/outputs/FRM_Analytics_Report.xlsx              ║
║  /content/outputs/frm_dashboard.html                     ║
║  /content/outputs/stress_results.csv                     ║
║  /content/charts/frm_dashboard.png                       ║
╚═════════════════════